# Temperature GGMP — Bundled Station-Fit Version

## 0) Config

Tweak these to trade off speed vs quality. For a quick smoke test, keep `FAST_MODE=True`.

GPU:
- Set `USE_GPU=True` to request fvGP GPU compute paths.
- If no compatible backend is present, GGMP will fall back to CPU.


## Important
This notebook prefers compact, full station-fit bundles in `temp_data/`.

Reuse-first behavior:
- if `temp_data/temperature_station_gmm_bundle_K{K}.npz` exists, the notebook loads it directly;
- if it is missing but `prefit/gmm_fits_*_K{K}.npz` exists and raw local files are available, the notebook reuses the prefit train cache and only materializes the missing test-side fits;
- observed train/test densities are reconstructed from bundled GMM fits rather than raw empirical histograms.


In [ ]:
from __future__ import annotations

import json
import os
import time
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Single source of truth: all helpers live in ggmp.py
# Notebook dev tip: reload ggmp so edits apply without a full kernel restart.
import importlib.util
import sys

_GGMP_MODULE_CANDIDATES = Path('ggmp.py')
_GGMP_MODULE_PATH = next((p for p in _GGMP_MODULE_CANDIDATES if p.exists()), None)
if _GGMP_MODULE_PATH is None:
    raise FileNotFoundError(
        'Could not find GGMP module. Expected one of: ggmp.py, ggmp 2.py'
    )
_spec = importlib.util.spec_from_file_location('ggmp', _GGMP_MODULE_PATH)
if _spec is None or _spec.loader is None:
    raise ImportError(f'Could not load GGMP module from {_GGMP_MODULE_PATH}')
ggmp_mod = importlib.util.module_from_spec(_spec)
sys.modules['ggmp'] = ggmp_mod
_spec.loader.exec_module(ggmp_mod)
logging.info('Loaded GGMP module from %s', _GGMP_MODULE_PATH.resolve())

from ggmp import (
    GGMP,
    hyperparameters,
    _get_key,
    gaussian_pdf,
    fit_station_gmms_fixed_weights_cached,
    build_gp_init_kwargs,
    _parse_device_ids,
    _normalize_pdf,
    train_component_gps_mcmc,
    prepare_station_terms_density,
    optimize_weights_em_density,
    bhattacharyya_distance,
    kl_divergence,
    wasserstein_1d,
)

# --------------------------
# Toggle-friendly parameters
# --------------------------
FAST_MODE = False
FIT_BUNDLE_OVERRIDE = None
BUILD_FROM_PREFIT_IF_NEEDED = True
FIT_BUNDLE_ROOT_CANDIDATES = [
    Path('temp_data'),
    Path('temperature_ggmp_upload_bundle/data'),
]
PREFIT_ROOT_CANDIDATES = [
    Path('prefit'),
    Path('temperature_ggmp_upload_bundle/prefit'),
]
RAW_DATA_ROOT_CANDIDATES = [
    Path('.'),
    Path('..'),
]
OUT_DIR = Path('ggmp_em_runs')

SEED = 42
MIN_SAMPLES = 50
TRAIN_RATIO = 0.8
K = 1  # choose any K that has a matching temperature_station_gmm_bundle_K{K}.npz file
OBSERVED_DENSITY_GRID_SIZE = 400 if FAST_MODE else 600
OBSERVED_STD_PAD = 4.0

# Station-level GMM fit settings for on-demand test-side materialization
GMM_MAX_ITER = 100
GMM_TOL = 1e-4
TEST_FIT_CACHE_DIR = Path('prefit_test_cache')

# GP MCMC iterations per component
MCMC_UNTIL_CONVERGED = True
MCMC_CHUNK = 10 if FAST_MODE else 100
MCMC_MAX_TOTAL = 50 if FAST_MODE else 5000
MCMC_TOL_REL = 1e-2 if FAST_MODE else 1e-3
MCMC_PATIENCE = 2 if FAST_MODE else 5

# Parallelize training across components (threads)
GP_PARALLEL = None
GP_WORKERS = None  # None => use K workers
BLAS_THREADS_PER_GP = 1  # recommended when GP_PARALLEL=True

# Save fvGP MCMC traces (gp.mcmc_info)
SAVE_GP_MCMC = True
GP_MCMC_THIN = 1
SAVE_GP_MCMC_CHUNKS = True  # when MCMC_UNTIL_CONVERGED=True

# If MCMC_UNTIL_CONVERGED=False, use a fixed budget:
N_UPDATES_GP = 50 if FAST_MODE else 500

# EM weight fit
WEIGHT_FLOOR = 1e-6
EM_MAX_ITER = 80 if FAST_MODE else 200
EM_TOL_L1 = 1e-10
EM_LOG_EVERY = 5 if FAST_MODE else 10

# Plotting
N_PLOT_STATIONS = 8
PLOT_GRID_SIZE = 300

# Threads (fvGP + BLAS)
N_THREADS = 1

# GPU options
USE_GPU = False
GPU_ENGINE = 'torch'   # 'torch' or 'cupy'
GP_DEVICE_IDS = None   # None, 'auto', '0,1', ...

# --------------------------
# Repro + logging
# --------------------------
np.random.seed(SEED)

os.environ['NTHREADS'] = str(int(N_THREADS))
os.environ['OPENBLAS_NUM_THREADS'] = str(int(N_THREADS))
os.environ['MKL_NUM_THREADS'] = str(int(N_THREADS))
os.environ['OMP_NUM_THREADS'] = str(int(N_THREADS))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S',
)

OUT_DIR.mkdir(parents=True, exist_ok=True)
run_dir = OUT_DIR / f'nb_run_{time.strftime("%Y%m%d_%H%M%S")}_K{K}'
run_dir.mkdir(parents=True, exist_ok=True)


def _load_npz_json(z, key='meta_json'):
    if key not in z.files:
        return {}
    return json.loads(str(z[key].tolist()))


def _resolve_existing_root(candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(candidates[0])


def _available_fit_bundle_paths(bundle_root: Path):
    root = Path(bundle_root)
    if not root.exists():
        return []
    return sorted(root.glob('temperature_station_gmm_bundle_K*.npz'))


def _available_prefit_paths(prefit_root: Path):
    root = Path(prefit_root)
    if not root.exists():
        return []
    return sorted(root.glob('gmm_fits_*_K*.npz'))


def _resolve_fit_bundle_path(bundle_root: Path, k: int, override=None) -> Path | None:
    if override is not None:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(f'FIT_BUNDLE_OVERRIDE does not exist: {path}')
        return path
    path = Path(bundle_root) / f'temperature_station_gmm_bundle_K{int(k)}.npz'
    if path.exists():
        return path
    return None


def _resolve_prefit_path(prefit_root: Path, k: int) -> Path | None:
    root = Path(prefit_root)
    candidates = sorted(root.glob(f'gmm_fits_*_K{int(k)}.npz'))
    return candidates[0] if candidates else None


def _station_domain_from_gmm(means, variances, *, n_grid: int, pad_std: float):
    means = np.asarray(means, dtype=float).reshape(-1)
    variances = np.asarray(variances, dtype=float).reshape(-1)
    variances = np.maximum(variances, 1e-12)
    std = np.sqrt(variances)
    lo = float(np.min(means - float(pad_std) * std))
    hi = float(np.max(means + float(pad_std) * std))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        center = float(np.mean(means)) if means.size else 0.0
        lo, hi = center - 5.0, center + 5.0
    return np.linspace(lo, hi, int(n_grid), dtype=float)


def _gmm_density(domain, means, variances, weights):
    domain = np.asarray(domain, dtype=float).reshape(-1)
    means = np.asarray(means, dtype=float).reshape(-1)
    variances = np.maximum(np.asarray(variances, dtype=float).reshape(-1), 1e-12)
    weights = np.asarray(weights, dtype=float).reshape(-1)
    if not (means.size == variances.size == weights.size):
        raise ValueError('means, variances, and weights must have the same length')
    mix = np.zeros_like(domain, dtype=float)
    for k in range(means.size):
        mix += float(weights[k]) * gaussian_pdf(domain, float(means[k]), float(variances[k]))
    _, mix, _ = _normalize_pdf(domain, mix)
    return mix


def _build_observed_density_list(means_mat, vars_mat, weights, *, n_grid: int, pad_std: float):
    out = []
    for means, vars_ in zip(np.asarray(means_mat, dtype=float), np.asarray(vars_mat, dtype=float)):
        domain = _station_domain_from_gmm(means, vars_, n_grid=int(n_grid), pad_std=float(pad_std))
        density = _gmm_density(domain, means, vars_, weights)
        out.append((domain, density))
    return out


def _extract_series_from_raw(temp, ids: np.ndarray):
    out = []
    for sid in np.asarray(ids, dtype=int):
        y = temp[:, int(sid)]
        y = y[~np.isnan(y)]
        out.append(np.asarray(y, dtype=float))
    return out


def _compute_split_from_raw(raw_root: Path):
    data_path = Path(raw_root) / 'data.npy'
    coord_path = Path(raw_root) / 'station_coord.npy'
    if not data_path.exists() or not coord_path.exists():
        raise FileNotFoundError(f'Expected raw files at {data_path} and {coord_path}')
    coords = np.load(coord_path)
    temp = np.load(data_path, mmap_mode='r')
    valid_counts = np.sum(~np.isnan(temp), axis=0)
    valid_station_ids = np.where(valid_counts >= int(MIN_SAMPLES))[0]
    rng = np.random.RandomState(int(SEED))
    n_train = int(len(valid_station_ids) * float(TRAIN_RATIO))
    perm = rng.permutation(len(valid_station_ids))
    train_idx = np.asarray(valid_station_ids[perm[:n_train]], dtype=int)
    test_idx = np.asarray(valid_station_ids[perm[n_train:]], dtype=int)
    lon = coords[:, 0]
    lat = coords[:, 1]
    x_train = np.column_stack([lon[train_idx], lat[train_idx]])
    x_test = np.column_stack([lon[test_idx], lat[test_idx]])
    return data_path, coord_path, temp, coords, valid_station_ids, train_idx, test_idx, x_train, x_test


def _materialize_bundle_from_prefit(prefit_path: Path, bundle_path: Path) -> Path:
    raw_root = next((Path(p) for p in RAW_DATA_ROOT_CANDIDATES if (Path(p) / 'data.npy').exists() and (Path(p) / 'station_coord.npy').exists()), None)
    if raw_root is None:
        raise FileNotFoundError('Missing raw data needed to materialize bundle from prefit. Expected data.npy and station_coord.npy locally.')

    data_path, coord_path, temp, coords, valid_station_ids, train_idx, test_idx, x_train, x_test = _compute_split_from_raw(raw_root)
    with np.load(prefit_path, allow_pickle=False) as z:
        prefit_meta = _load_npz_json(z)
        prefit_station_ids = np.asarray(z['station_ids'], dtype=int)
        prefit_means = np.asarray(z['means'], dtype=float)
        prefit_vars = np.asarray(z['vars'], dtype=float)

    if int(prefit_meta.get('K', K)) != int(K):
        raise ValueError(f'Prefit K mismatch for {prefit_path}: {prefit_meta.get("K")} vs notebook K={int(K)}')
    if not np.array_equal(np.sort(prefit_station_ids), np.sort(train_idx)):
        raise ValueError('Prefit station ids do not match the expected train split for this notebook configuration.')

    order = {int(sid): i for i, sid in enumerate(prefit_station_ids)}
    train_means = np.vstack([prefit_means[order[int(sid)]] for sid in train_idx])
    train_vars = np.vstack([prefit_vars[order[int(sid)]] for sid in train_idx])

    test_series = _extract_series_from_raw(temp, test_idx)
    test_means, test_vars, test_cache_path = fit_station_gmms_fixed_weights_cached(
        test_series,
        test_idx,
        data_path=data_path,
        K=int(K),
        gmm_max_iter=int(GMM_MAX_ITER),
        gmm_tol=float(GMM_TOL),
        cache=True,
        cache_dir=Path(TEST_FIT_CACHE_DIR),
        log_every=250,
        logger=logging.getLogger('temp_ggmp_bundle'),
    )

    bundle_meta = {
        'dataset': 'temperature_extremes',
        'K': int(K),
        'seed': int(SEED),
        'min_samples': int(MIN_SAMPLES),
        'train_ratio': float(TRAIN_RATIO),
        'n_valid_stations': int(len(valid_station_ids)),
        'n_train': int(len(train_idx)),
        'n_test': int(len(test_idx)),
        'gmm_max_iter': int(GMM_MAX_ITER),
        'gmm_tol': float(GMM_TOL),
        'source_data_file': str(data_path),
        'source_coords_file': str(coord_path),
        'source_prefit_path': str(prefit_path),
        'test_cache_path': str(test_cache_path) if test_cache_path is not None else None,
        'notes': 'Train fits loaded from prefit cache; test fits materialized locally to complete the upload bundle.',
    }

    bundle_path = Path(bundle_path)
    bundle_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        bundle_path,
        coords=np.asarray(coords, dtype=np.float64),
        valid_station_ids=np.asarray(valid_station_ids, dtype=np.int64),
        train_station_ids=np.asarray(train_idx, dtype=np.int64),
        test_station_ids=np.asarray(test_idx, dtype=np.int64),
        x_train=np.asarray(x_train, dtype=np.float64),
        x_test=np.asarray(x_test, dtype=np.float64),
        train_means=np.asarray(train_means, dtype=np.float64),
        train_vars=np.asarray(train_vars, dtype=np.float64),
        test_means=np.asarray(test_means, dtype=np.float64),
        test_vars=np.asarray(test_vars, dtype=np.float64),
        meta_json=json.dumps(bundle_meta, sort_keys=True),
    )
    logging.info('Materialized bundle from prefit: %s', bundle_path)
    return bundle_path


logging.info('run_dir=%s', run_dir)
logging.info(
    'FAST_MODE=%s | K=%d | MCMC_UNTIL_CONVERGED=%s | N_UPDATES_GP=%s | MCMC_CHUNK=%s | MCMC_MAX_TOTAL=%s',
    FAST_MODE,
    K,
    MCMC_UNTIL_CONVERGED,
    N_UPDATES_GP,
    MCMC_CHUNK,
    MCMC_MAX_TOTAL,
)


## 1) Load bundled station-fit data

In [ ]:
FIT_BUNDLE_ROOT = _resolve_existing_root(FIT_BUNDLE_ROOT_CANDIDATES)
PREFIT_ROOT = _resolve_existing_root(PREFIT_ROOT_CANDIDATES)
available_bundle_paths = _available_fit_bundle_paths(FIT_BUNDLE_ROOT)
available_prefit_paths = _available_prefit_paths(PREFIT_ROOT)
logging.info('Fit bundle root=%s', FIT_BUNDLE_ROOT.resolve())
logging.info('Prefit root=%s', PREFIT_ROOT.resolve())
logging.info('Available fit bundles: %s', [p.name for p in available_bundle_paths])
logging.info('Available prefit files: %s', [p.name for p in available_prefit_paths])

fit_bundle_path = _resolve_fit_bundle_path(FIT_BUNDLE_ROOT, int(K), FIT_BUNDLE_OVERRIDE)
if fit_bundle_path is None:
    prefit_path = _resolve_prefit_path(PREFIT_ROOT, int(K))
    if prefit_path is None:
        raise FileNotFoundError(
            f'No full bundle or prefit file found for K={int(K)}. '
            f'Available bundles: {[p.name for p in available_bundle_paths]} | '
            f'Available prefits: {[p.name for p in available_prefit_paths]}'
        )
    if not bool(BUILD_FROM_PREFIT_IF_NEEDED):
        raise FileNotFoundError(
            f'Bundle missing for K={int(K)} at {FIT_BUNDLE_ROOT}, and BUILD_FROM_PREFIT_IF_NEEDED=False.'
        )
    fit_bundle_path = Path(FIT_BUNDLE_ROOT) / f'temperature_station_gmm_bundle_K{int(K)}.npz'
    fit_bundle_path = _materialize_bundle_from_prefit(prefit_path, fit_bundle_path)

with np.load(fit_bundle_path, allow_pickle=False) as z:
    fit_bundle_meta = _load_npz_json(z)
    coords = np.asarray(z['coords'], dtype=float)
    valid_station_ids = np.asarray(z['valid_station_ids'], dtype=int)
    train_idx = np.asarray(z['train_station_ids'], dtype=int)
    test_idx = np.asarray(z['test_station_ids'], dtype=int)
    x_train = np.asarray(z['x_train'], dtype=float)
    x_test = np.asarray(z['x_test'], dtype=float)
    train_means_mat = np.asarray(z['train_means'], dtype=float)
    train_vars_mat = np.asarray(z['train_vars'], dtype=float)
    test_means_mat = np.asarray(z['test_means'], dtype=float)
    test_vars_mat = np.asarray(z['test_vars'], dtype=float)

bundle_k = int(fit_bundle_meta.get('K', K))
if bundle_k != int(K):
    raise ValueError(f'Notebook K={int(K)} does not match bundle K={bundle_k}: {fit_bundle_path}')

expected_min_samples = int(fit_bundle_meta.get('min_samples', MIN_SAMPLES))
expected_train_ratio = float(fit_bundle_meta.get('train_ratio', TRAIN_RATIO))
if expected_min_samples != int(MIN_SAMPLES):
    raise ValueError(f'MIN_SAMPLES mismatch: notebook={MIN_SAMPLES} bundle={expected_min_samples}')
if abs(expected_train_ratio - float(TRAIN_RATIO)) > 1e-12:
    raise ValueError(f'TRAIN_RATIO mismatch: notebook={TRAIN_RATIO} bundle={expected_train_ratio}')

assert train_means_mat.shape == train_vars_mat.shape
assert test_means_mat.shape == test_vars_mat.shape
assert train_means_mat.shape[1] == int(K)
assert test_means_mat.shape[1] == int(K)
assert x_train.shape[0] == train_idx.size == train_means_mat.shape[0]
assert x_test.shape[0] == test_idx.size == test_means_mat.shape[0]

w_init = np.ones(int(K), dtype=float) / float(K)
y_data_train = _build_observed_density_list(
    train_means_mat,
    train_vars_mat,
    w_init,
    n_grid=int(OBSERVED_DENSITY_GRID_SIZE),
    pad_std=float(OBSERVED_STD_PAD),
)

logging.info('Loaded bundled station fits: %s', fit_bundle_path)
logging.info(
    'Stations | valid=%d | train=%d | test=%d | bundle_dataset=%s',
    len(valid_station_ids),
    len(train_idx),
    len(test_idx),
    fit_bundle_meta.get('dataset', 'unknown'),
)
logging.info(
    'Fit matrices | train=%s | test=%s',
    train_means_mat.shape,
    test_means_mat.shape,
)


In [ ]:
# Leakage sanity checks (pre-model)
assert np.intersect1d(train_idx, test_idx).size == 0
assert x_train.shape[0] == len(train_idx)
assert x_test.shape[0] == len(test_idx)
logging.info('Leakage checks passed: bundled train/test station ids are disjoint.')


## 2) Load station mixtures (precomputed fixed weights)

In [ ]:
w_init = np.ones(int(K), dtype=float) / float(K)
means_mat = np.asarray(train_means_mat, dtype=float)
vars_mat = np.asarray(train_vars_mat, dtype=float)

logging.info('Using precomputed station GMM fits (no raw temperature samples loaded in the notebook).')
logging.info('Train fit arrays | means=%s | vars=%s', means_mat.shape, vars_mat.shape)
logging.info('Test fit arrays  | means=%s | vars=%s', test_means_mat.shape, test_vars_mat.shape)

means_mat.shape, vars_mat.shape


### 2a) Train-station fit verification plot (bundled marginals)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

sample_indices = np.random.choice(len(train_idx), size=min(int(N_PLOT_STATIONS), len(train_idx)), replace=False)

for plot_idx, i in enumerate(sample_indices):
    ax = axes[plot_idx]

    domain, p_obs = y_data_train[i]
    means = means_mat[i]
    vars_ = vars_mat[i]

    ax.plot(domain, p_obs, color='k', linewidth=2, label='Observed fit')

    gmm_pdf = np.zeros_like(domain)
    for k in range(int(K)):
        component_pdf = float(w_init[k]) * gaussian_pdf(domain, float(means[k]), float(vars_[k]))
        gmm_pdf += component_pdf
        ax.plot(domain, component_pdf, '--', alpha=0.5, label=f'Comp {k}')

    ax.plot(domain, gmm_pdf, 'r-', linewidth=2, label='GMM fit')
    ax.set_title(f'Station {int(train_idx[i])}')
    ax.legend(fontsize=6)

plt.tight_layout()
plt.show()


## 3) Train GPs (fvGP MCMC)

In [ ]:
# GGMP hyperparameters setup
weights_bounds = np.array([[float(WEIGHT_FLOOR), 1.0]] * int(K), dtype=float)

hps_list = []
hps_bounds_list = []
for k in range(int(K)):
    mean_k = float(np.mean(means_mat[:, k]))
    hps_list.append(np.array([1.0, 1.0, 1.0, mean_k], dtype=float))  # [signal, L_lon, L_lat, mean]
    hps_bounds_list.append(
        np.array(
            [
                [0.1, 10.0],
                [0.01, 100.0],
                [0.01, 100.0],
                [mean_k - 40.0, mean_k + 40.0],
            ],
            dtype=float,
        )
    )

hps_obj = hyperparameters(w_init.copy(), weights_bounds, hps_list, hps_bounds_list)

# fvGP GPU toggles (passed via GGMP.initGPs)
gp_init_kwargs, _ = build_gp_init_kwargs(use_gpu=USE_GPU, gpu_engine=GPU_ENGINE)

model = GGMP(
    x_train,
    y_data_train,
    hps_obj=hps_obj,
    likelihood_terms=int(K),
    gp_init_kwargs=gp_init_kwargs,
    gp_device_ids=_parse_device_ids(GP_DEVICE_IDS),
)

init_mean = [means_mat[:, k] for k in range(int(K))]
init_std = [np.sqrt(vars_mat[:, k]) for k in range(int(K))]
model.initLikelihoods(init_mean=init_mean, init_std=init_std, weights=w_init)
model.initGPs()

logging.info('Initialized GGMP from bundled station fits: N=%d, K=%d', model.len_data, model.likelihood_terms)

# Leakage sanity checks (post-model)
assert model.x_data.shape == x_train.shape and np.allclose(model.x_data, x_train)
assert model.len_data == len(train_idx)
assert len(model.y_data) == len(train_idx)
logging.info('Leakage checks passed: GGMP model is built only from bundled train-station fits.')


In [ ]:
trained_hps = train_component_gps_mcmc(
    model,
    hps_obj,
    n_updates_gp=int(N_UPDATES_GP),
    mcmc_until_converged=bool(MCMC_UNTIL_CONVERGED),
    mcmc_chunk=int(MCMC_CHUNK),
    mcmc_max_total=int(MCMC_MAX_TOTAL),
    mcmc_tol_rel=float(MCMC_TOL_REL),
    mcmc_patience=int(MCMC_PATIENCE),
    gp_parallel=bool(GP_PARALLEL),
    gp_workers=(int(GP_WORKERS) if GP_WORKERS is not None else None),
    blas_threads_per_gp=(int(BLAS_THREADS_PER_GP) if BLAS_THREADS_PER_GP is not None else None),
    run_dir=run_dir,
    save_gp_mcmc=bool(SAVE_GP_MCMC),
    gp_mcmc_thin=int(GP_MCMC_THIN),
    save_gp_mcmc_chunks=bool(SAVE_GP_MCMC_CHUNKS),
)

trained_hps


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Optional override if you want a specific run directory.
RUN_DIR_OVERRIDE = None  # e.g. Path("ggmp_em_runs/nb_run_YYYYMMDD_HHMMSS_K1")

base_runs = OUT_DIR if 'OUT_DIR' in globals() else Path('ggmp_em_runs')
if RUN_DIR_OVERRIDE is not None:
    RUN_DIR = Path(RUN_DIR_OVERRIDE)
else:
    candidates = sorted(base_runs.glob('nb_run_*_K*'))
    if not candidates:
        raise FileNotFoundError(f'No run directories found under {base_runs}')
    RUN_DIR = candidates[-1]

BURN_FRAC = 0.3

def _load_npz(p: Path):
    z = np.load(p, allow_pickle=False)
    x = np.asarray(z["x"], dtype=float)
    meta = json.loads(str(z["meta_json"])) if "meta_json" in z.files else {}
    fx = np.asarray(z["f"], dtype=float) if "f" in z.files else None
    return x, fx, meta

def _plot_trace(k: int, X: np.ndarray, FX=None, title_extra=""):
    n, d = X.shape
    burn = int(BURN_FRAC * n)

    nrows = d + (1 if FX is not None else 0)
    fig, axes = plt.subplots(nrows, 1, figsize=(11, 2.2 * nrows), sharex=True)
    axes = np.atleast_1d(axes)

    for j in range(d):
        axes[j].plot(X[:, j], lw=0.8)
        axes[j].axvline(burn, color="r", lw=1, alpha=0.6)
        axes[j].set_ylabel(f"hps[{j}]")
        axes[j].grid(True, alpha=0.25)

    if FX is not None:
        axes[-1].plot(FX, lw=0.8, color="k")
        axes[-1].axvline(burn, color="r", lw=1, alpha=0.6)
        axes[-1].set_ylabel("f(x)")
        axes[-1].grid(True, alpha=0.25)

    axes[-1].set_xlabel("sample")
    fig.suptitle(f"GP[{k}] MCMC trace {title_extra} | samples={n} dim={d}", y=0.98)
    plt.tight_layout()
    plt.show()

print("RUN_DIR:", RUN_DIR.resolve())

trace_files = sorted(RUN_DIR.glob("gp*_mcmc_trace*.npz")) + sorted((RUN_DIR / "gp_mcmc").glob("gp*_mcmc_trace*.npz"))
trace_files = sorted({p.resolve() for p in trace_files})

print("Found trace files:", len(trace_files))
for p in trace_files[:30]:
    print(" -", p.name)

by_k = {}
for p in trace_files:
    try:
        _, _, meta = _load_npz(p)
        k = int(meta.get("k"))
    except Exception:
        k = int(p.name[p.name.find("gp")+2:p.name.find("gp")+4])
    by_k.setdefault(k, []).append(p)

for k, files in sorted(by_k.items()):
    chunk_files = sorted([p for p in files if "chunk" in p.name])
    if chunk_files:
        Xs, FXs = [], []
        for p in chunk_files:
            X, FX, _ = _load_npz(p)
            Xs.append(X)
            if FX is not None:
                FXs.append(FX)
        Xfull = np.vstack(Xs)
        FXfull = np.concatenate(FXs) if (FXs and len(FXs) == len(Xs)) else None
        _plot_trace(k, Xfull, FXfull, title_extra="(concatenated chunks)")
    else:
        p = sorted(files)[-1]
        X, FX, _ = _load_npz(p)
        _plot_trace(k, X, FX, title_extra=f"({p.name})")


## 4) EM weight optimization (density objective)

We optimize shared weights \(w\) for the density-based Eq. 11 objective:

$$
\sum_{i=1}^N \int p_{	ext{obs},i}(y)\; \log\Big( \sum_{k=1}^K w_k\;\mathcal{N}(y\mid \mu_{ik},\,\sigma^2_{ik}) \Big)\;dy
$$

We approximate each integral by a discrete sum on a histogram grid, and run EM on \(w\).


In [ ]:
terms, ll_comp = prepare_station_terms_density(model, trained_hps)
logging.info("Expected component-only log-likelihoods: %s", np.array2string(ll_comp, precision=3))

w_em, w_hist, obj_hist = optimize_weights_em_density(
    terms,
    K=int(K),
    weight_floor=float(WEIGHT_FLOOR),
    max_iter=int(EM_MAX_ITER),
    tol_l1=float(EM_TOL_L1),
    log_every=int(EM_LOG_EVERY),
    w0=w_init,
)

logging.info("Final EM weights: %s", np.array2string(w_em, precision=6))
logging.info("Final objective: %.6f", float(obj_hist[-1]) if obj_hist.size else float('nan'))

# Plot traces
fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(obj_hist, '-k')
ax1.set_title('EM objective')
ax1.set_xlabel('iteration')
ax1.set_ylabel('objective')
ax2 = fig.add_subplot(1, 2, 2)
for k in range(int(K)):
    ax2.plot(w_hist[:, k], label=f'w[{k}]')
ax2.set_title('EM weights')
ax2.set_xlabel('iteration')
ax2.set_ylabel('weight')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 4a) Marginalization verification plot (using optimized weights)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

sample_indices = np.random.choice(len(train_idx), size=min(int(N_PLOT_STATIONS), len(train_idx)), replace=False)

for plot_idx, i in enumerate(sample_indices):
    ax = axes[plot_idx]

    domain, p_obs = y_data_train[i]
    means = means_mat[i]
    vars_ = vars_mat[i]

    ax.plot(domain, p_obs, color='k', linewidth=2, label='Observed fit')

    gmm_pdf = np.zeros_like(domain)
    for k in range(int(K)):
        component_pdf = float(w_em[k]) * gaussian_pdf(domain, float(means[k]), float(vars_[k]))
        gmm_pdf += component_pdf
        ax.plot(domain, component_pdf, '--', alpha=0.5, label=f'Comp {k}')

    ax.plot(domain, gmm_pdf, 'r-', linewidth=2, label='Weighted GMM fit')
    ax.set_title(f'Station {int(train_idx[i])}')
    ax.legend(fontsize=6)

plt.tight_layout()
plt.show()


## 5) Test evaluation against bundled station-fit densities

In [ ]:
# Update model weights so posterior mixing uses w_em
model.hps_obj.set(w_em, trained_hps)
for k in range(int(K)):
    try:
        model.likelihoods[k].set_weight(float(w_em[k]))
    except Exception:
        pass

# GP predictive mean/variance for component means at test stations
mu_test = np.empty((len(test_idx), int(K)), dtype=float)
var_gp_test = np.empty((len(test_idx), int(K)), dtype=float)

for k in range(int(K)):
    gp = model.gps[k]
    model._safe_set_hyperparameters(gp, trained_hps[k])
    pm = gp.posterior_mean(x_test)
    pc = gp.posterior_covariance(x_test, variance_only=True)
    mu_test[:, k] = np.asarray(_get_key(pm, ('m(x)', 'f(x)')), dtype=float).reshape(-1)
    var_gp_test[:, k] = np.maximum(np.asarray(_get_key(pc, ('v(x)', 'v', 'variance', 'cov')), dtype=float).reshape(-1), 0.0)

# Use global within-component variance (mean over train stations)
var_comp_global = np.array([float(np.mean(model.likelihoods[k].variance)) for k in range(int(K))], dtype=float)
var_comp_global = np.maximum(var_comp_global, 1e-9)

test_density_cache = []
metrics = []
for j in range(len(test_idx)):
    obs_means = np.asarray(test_means_mat[j], dtype=float)
    obs_vars = np.asarray(test_vars_mat[j], dtype=float)
    pred_means = np.asarray(mu_test[j, :], dtype=float)
    pred_vars = np.asarray(var_gp_test[j, :] + var_comp_global[:], dtype=float)
    pred_vars = np.maximum(pred_vars, 1e-12)

    domain = _station_domain_from_gmm(
        np.concatenate([obs_means, pred_means]),
        np.concatenate([obs_vars, pred_vars]),
        n_grid=int(OBSERVED_DENSITY_GRID_SIZE),
        pad_std=float(OBSERVED_STD_PAD),
    )
    p_obs = _gmm_density(domain, obs_means, obs_vars, w_init)
    mix = _gmm_density(domain, pred_means, pred_vars, w_em)

    domain, p_obs, dx = _normalize_pdf(domain, p_obs)
    _, mix, _ = _normalize_pdf(domain, mix)

    bdist = bhattacharyya_distance(domain, p_obs, mix)
    kl_pq = kl_divergence(domain, p_obs, mix)
    kl_qp = kl_divergence(domain, mix, p_obs)
    w1 = wasserstein_1d(domain, p_obs, mix)
    l1 = float(np.sum(np.abs(p_obs - mix) * dx))

    metrics.append({
        'station_id': int(test_idx[j]),
        'bhattacharyya': float(bdist),
        'kl_sym': float(0.5 * (kl_pq + kl_qp)),
        'wasserstein1': float(w1),
        'l1': float(l1),
    })
    test_density_cache.append({'domain': domain, 'p_obs': p_obs, 'p_pred': mix})

# Summaries
for key in ('bhattacharyya', 'kl_sym', 'wasserstein1', 'l1'):
    arr = np.asarray([m[key] for m in metrics], dtype=float)
    logging.info('TEST %s: mean=%.4f | median=%.4f | std=%.4f', key, float(np.mean(arr)), float(np.median(arr)), float(np.std(arr)))

metrics[:3]


In [ ]:
# Metric histograms
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()
keys = ['bhattacharyya', 'kl_sym', 'wasserstein1', 'l1']

for ax, key in zip(axes, keys):
    vals = np.asarray([m[key] for m in metrics], dtype=float)
    ax.hist(vals, bins=30, alpha=0.8)
    ax.set_title(f"Test {key}")

plt.tight_layout()
plt.show()


In [ ]:
# Plot a few test stations: observed bundled fit vs predicted mixture
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

if len(test_idx) > 0:
    sample_test = np.random.choice(len(test_idx), size=min(int(N_PLOT_STATIONS), len(test_idx)), replace=False)
    for plot_idx, j in enumerate(sample_test):
        ax = axes[plot_idx]
        cached = test_density_cache[j]
        domain = np.asarray(cached['domain'], dtype=float)
        p_obs = np.asarray(cached['p_obs'], dtype=float)
        p_pred = np.asarray(cached['p_pred'], dtype=float)

        ax.plot(domain, p_obs, color='k', linewidth=2, label='Observed fit')

        for k in range(int(K)):
            sigma = float(np.sqrt(max(var_gp_test[j, k] + var_comp_global[k], 1e-12)))
            comp_pdf = float(w_em[k]) * norm.pdf(domain, loc=float(mu_test[j, k]), scale=sigma)
            ax.plot(domain, comp_pdf, '--', alpha=0.5, label=f'Comp {k}')

        ax.plot(domain, p_pred, 'r-', linewidth=2, label='Pred Mix')
        ax.set_title(f'Test Station {int(test_idx[j])}')
        ax.legend(fontsize=6)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import json

plot_js = [55, 22, 34, 15]
plot_js = [j for j in plot_js if j < len(test_idx)]

payload = {
    "K": int(K),
    "test_js": plot_js,
    "test_idx_vals": [int(test_idx[j]) for j in plot_js],
    "components": {}
}

for j in plot_js:
    wk = np.asarray(w_em, dtype=float)  # (K,)
    muk = np.asarray(mu_test[j, :], dtype=float)  # (K,)
    var_total = np.asarray(var_gp_test[j, :] + var_comp_global[:], dtype=float)  # (K,)
    var_total = np.maximum(var_total, 1e-12)

    payload["components"][str(j)] = {
        "w": wk.tolist(),
        "mu": muk.tolist(),
        "var": var_total.tolist(),
    }

print(json.dumps(payload, indent=2))


## 6) Save artifacts (optional)

In [ ]:
import csv

# Save EM traces
np.save(run_dir / 'weights_em_history.npy', w_hist)
np.save(run_dir / 'objective_em_history.npy', obj_hist)

# Save fit-bundle provenance
(run_dir / 'fit_bundle_path.txt').write_text(str(Path(fit_bundle_path).resolve()) + '\n')
(run_dir / 'fit_bundle_meta.json').write_text(json.dumps(fit_bundle_meta, indent=2, sort_keys=True) + '\n')

# Save test metrics
metrics_path = run_dir / 'test_metrics.csv'
with open(metrics_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(metrics[0].keys()))
    writer.writeheader()
    writer.writerows(metrics)

logging.info('Saved %s', metrics_path)
